# Databricks notebook source

In [0]:
BASE_PATH = "/Volumes/workspace/default/university_chapters"

BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold/v1"
QUARANTINE_PATH = f"{BASE_PATH}/quarantine"

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.university_chapters

In [0]:
%sql
--LIST '/Volumes/workspace/default/university_chapters'

In [0]:
API_BASE_URL = (
    "https://services2.arcgis.com/5I7u4SJE1vUr79JC/arcgis/rest/services/"
    "UniversityChapters_Public/FeatureServer/0/query"
)

STATES = ["CA", "OR", "WA"]

In [0]:
API_QUERY_PARAMS = {
    "where": f"State IN ({','.join(repr(s) for s in STATES)})",
    "outFields": "*",
    "returnGeometry": "true",
    "f": "json",
}

#print(API_QUERY_PARAMS)

In [0]:
COLUMN_MAPPING = {
    "ChapterID": "chapter_id",
    "University_Chapter": "chapter_name",
    "City": "city",
    "State": "state",
    "OBJECTID": "source_object_id",
}

In [0]:
GOLD_COLUMNS = [
    "chapter_id",
    "chapter_name",
    "city",
    "state",
    "longitude",
    "latitude",
    "dq_status",
    "dq_warnings",
]

In [0]:
DQ_REASON_INVALID_COORDINATES = "INVALID_COORDINATES"
DQ_REASON_MISSING_UNKNOWN_CITY = "MISSING_OR_UNKNOWN_CITY"

DQ_STATUS_OK = "OK"
DQ_STATUS_WARNING = "WARNING"

"history by run" — folders full of past snapshots.

In [0]:
#Way to name each pipeline runs
#When 01_ingest_bronze.py runs new_run_id(), it gets a text string back (e.g. 20260913T143022Z_a1b2c3d4)
#It then uses that string as a folder name when it writes the actual JSON data, e.g.:

import uuid
from datetime import datetime, timezone

def new_run_id() -> str:
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    short_uuid = uuid.uuid4().hex[:8] 
    return f"{ts}_{short_uuid}"

In [0]:
#find the latest one - most recent
#It checks - "what folders exist right now?"

def latest_bronze_run_path() -> str:
    runs = [f.name.rstrip("/") for f in dbutils.fs.ls(BRONZE_PATH)]
    if not runs:
        raise RuntimeError(f"No bronze runs found under {BRONZE_PATH}")
    latest = sorted(runs)[-1]
    return f"{BRONZE_PATH}/{latest}"